In [ ]:
!nvidia-smi

Tue Nov 24 20:59:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("ERROR: GPU not detected. Check Runtime → Change runtime type → T4 GPU.")


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU name: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/BR_repro_magmap_colab.zip'
EXPECTED_SHA256 = '34702b057a25f05a18f10a834150ccf49b19fa06d58be062a4f8361fbeb7db6b'
import hashlib, os
h = hashlib.sha256(open(ZIP, 'rb').read()).hexdigest()
print('zip sha256', h, 'OK' if h == EXPECTED_SHA256 else 'MISMATCH - re-upload the zip')
!rm -rf /content/BR && unzip -q "$ZIP" -d /content && ls /content/BR /content/BR/Data/RFI-Patch
RUNS = '/content/drive/MyDrive/repro_magmap_runs'
os.makedirs(RUNS, exist_ok=True)
%cd /content/BR/sar_rfi_baseline

Mounted at /content/drive
zip sha256 34702b057a25f05a18f10a834150ccf49b19fa06d58be062a4f8361fbeb7db6b OK
/content/BR:
Data  sar_rfi_baseline	SAR-RFI-DATASET-main

/content/BR/Data/RFI-Patch:
explaination.docx  ReadPatch.m	RFI-1
/content/BR/sar_rfi_baseline


In [ ]:
!pip -q install pyyaml scikit-image
!python -m pytest repro_magmap/tests -q

.....................                                                    [100%]
=============================== warnings summary ===============================
repro_magmap/tests/test_magmap.py::test_identical_to_authors_class
  /usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
    warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
21 passed, 1 warning in 12.96s


In [ ]:
# checks 1-7 (data, pairing, dims, STFT/ISTFT, normalization, U-Net = authors' class)
import os
if not os.path.exists(f'{RUNS}/verification_colab'):
    !python repro_magmap/scripts/verify_pipeline.py --out-dir {RUNS}/verification_colab

In [ ]:
# checks 8-9 with a longer budget than the CPU run (one -20 dB training map, fp32, authors' optimizer settings)
if not os.path.exists(f'{RUNS}/overfit_gpu'):
    !python repro_magmap/scripts/overfit_one_sample.py --out-dir {RUNS}/overfit_gpu --steps 3000 --eval-every 100 --device cuda --max-minutes 30

In [ ]:
# FP32 vs AMP before the full run (same init, same 300 batches); writes the decision
import json, os
if not os.path.exists(f'{RUNS}/amp_check/amp_comparison.json'):
    !python repro_magmap/scripts/compare_amp.py --out-dir {RUNS}/amp_check --steps 300
PRECISION = json.load(open(f'{RUNS}/amp_check/amp_comparison.json'))['summary']['recommendation']
print('precision for the full run:', PRECISION)

precision for the full run: fp32


In [ ]:
import os, json

p = f'{RUNS}/overfit_gpu/overfit_result.json'

print("Overfit result exists:", os.path.exists(p))

if os.path.exists(p):
    result = json.load(open(p))
    print("Overfit passed:", result['pass'])

print("Precision:", PRECISION)
print("Training output:", f'{RUNS}/unet_official_{PRECISION}')

Overfit result exists: True
Overfit passed: True
Precision: fp32
Training output: /content/drive/MyDrive/repro_magmap_runs/unet_official_fp32


In [ ]:
import os, torch

RUN = f'{RUNS}/unet_official_{PRECISION}'
CKPT = f'{RUN}/checkpoints/last.pt'

print("Checkpoint exists:", os.path.exists(CKPT))

if os.path.exists(CKPT):
    print("Checkpoint size (MB):", round(os.path.getsize(CKPT)/1024**2, 2))
    print("Last modified:", __import__('datetime').datetime.fromtimestamp(
        os.path.getmtime(CKPT)
    ))

    ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)

    print("Checkpoint keys:", ckpt.keys())
    for key in ckpt.keys():
        if 'epoch' in key.lower() or 'step' in key.lower():
            print(key, "=", ckpt[key])
else:
    raise RuntimeError("STOP: No checkpoint found. Do NOT start training.")

Checkpoint exists: True
Checkpoint size (MB): 108.86
Last modified: 2026-11-24 15:47:19
Checkpoint keys: dict_keys(['epoch', 'state_dict', 'optimizer', 'scheduler', 'scaler', 'config', 'map_config', 'normalization', 'precision_used', 'best_val_psnr', 'history', 'rng', 'loader_generator'])
epoch = 100


In [ ]:
# RUN = f'{RUNS}/unet_official_{PRECISION}'
# CKPT = f'{RUN}/checkpoints/last.pt'

# assert os.path.exists(CKPT), "Checkpoint missing! Training cannot resume."

# resume = '--resume'

# !python repro_magmap/scripts/train.py --out-dir {RUN} --device cuda --precision {PRECISION} {resume}

In [ ]:
import torch

ckpt = torch.load(
    f'{RUN}/checkpoints/last.pt',
    map_location='cpu',
    weights_only=False
)

print("Saved epoch:", ckpt['epoch'])
print("RNG state type:", type(ckpt['rng']['torch']))
print("RNG state dtype:", ckpt['rng']['torch'].dtype)
print("RNG state device:", ckpt['rng']['torch'].device)
print("RNG state shape:", ckpt['rng']['torch'].shape)

Saved epoch: 100
RNG state type: <class 'torch.Tensor'>
RNG state dtype: torch.uint8
RNG state device: cpu
RNG state shape: torch.Size([5056])


In [ ]:
!sed -n '125,155p' repro_magmap/magmap/trainer.py

                from .metrics8 import psnr_u8, to_u8

                mx = float(t.max())
                psnr.append(psnr_u8(to_u8(o, mx)[0], to_u8(t, mx)[0]))
            else:
                psnr.append(m["psnr"])
                ssim.append(m["ssim"])
    fin = [p for p in psnr if np.isfinite(p)]
    return {"val_loss": float(np.mean(losses)) if losses else float("nan"),
            "val_psnr": float(np.mean(fin)) if fin else float("nan"),
            "val_ssim": float(np.mean(ssim)) if ssim else None, "n_maps": len(psnr)}


def rng_state() -> dict:
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s: dict) -> None:
    random.setstate(s["python"])
    np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all

In [ ]:
from pathlib import Path

path = Path("repro_magmap/magmap/trainer.py")
code = path.read_text()

old = 'torch.set_rng_state(s["torch"])'
new = 'torch.set_rng_state(s["torch"].cpu())'

assert old in code, "Original line not found. No changes made."

path.write_text(code.replace(old, new, 1))

print("Successfully patched RNG restoration.")

Successfully patched RNG restoration.


In [ ]:
!grep -n -A 8 "def set_rng_state" repro_magmap/magmap/trainer.py

143:def set_rng_state(s: dict) -> None:
144-    random.setstate(s["python"])
145-    np.random.set_state(s["numpy"])
146-    torch.set_rng_state(s["torch"].cpu())
147-    if s.get("cuda") is not None and torch.cuda.is_available():
148-        torch.cuda.set_rng_state_all(s["cuda"])


In [ ]:
!sed -n '130,165p' repro_magmap/scripts/train.py

    train_ds = P.TorchPairs(P.MapPairs(idx, parts["train"], mcfg, norm))
    val_ds = P.TorchPairs(P.MapPairs(idx, parts["val"], mcfg, norm))
    gen = torch.Generator()
    gen.manual_seed(t["seed"])
    nw = t["num_workers"]
    train_loader = DataLoader(train_ds, batch_size=t["batch_size"], shuffle=t["shuffle"], drop_last=t["drop_last"],
                              num_workers=nw, generator=gen, pin_memory=device.type == "cuda",
                              persistent_workers=nw > 0)
    val_loader = DataLoader(val_ds, batch_size=t["batch_size"], shuffle=False, num_workers=nw,
                            pin_memory=device.type == "cuda", persistent_workers=nw > 0)

    x0, y0, _ = train_ds[0]
    log(f"[check] sample tensors {tuple(x0.shape)} {x0.dtype}; input max {x0.max():.4f}; label max {y0.max():.4f}")
    with torch.no_grad():
        o0 = model(x0[None].to(device))
    assert o0.shape == (1,) + tuple(x0.shape), o0.shape

    start_epoch, best_psnr, history = 1, -np.inf, []


In [ ]:
from pathlib import Path

path = Path("repro_magmap/scripts/train.py")
code = path.read_text()

old = 'torch.load(ck / "last.pt", map_location=device, weights_only=False)'
new = 'torch.load(ck / "last.pt", map_location="cpu", weights_only=False)'

assert old in code, "Expected checkpoint-loading line not found."

path.write_text(code.replace(old, new, 1))

print("Checkpoint loading patched successfully.")

Checkpoint loading patched successfully.


In [ ]:
!grep -n 'torch.load(ck' repro_magmap/scripts/train.py
!grep -n -A 5 'def set_rng_state' repro_magmap/magmap/trainer.py

150:        s = torch.load(ck / "last.pt", map_location="cpu", weights_only=False)
143:def set_rng_state(s: dict) -> None:
144-    random.setstate(s["python"])
145-    np.random.set_state(s["numpy"])
146-    torch.set_rng_state(s["torch"].cpu())
147-    if s.get("cuda") is not None and torch.cuda.is_available():
148-        torch.cuda.set_rng_state_all(s["cuda"])


In [ ]:
# full training (100 epochs); re-run this cell after a disconnect: it resumes from last.pt
import json, os
assert json.load(open(f'{RUNS}/overfit_gpu/overfit_result.json'))['pass'], 'one-sample overfit did not pass: do not start the full run'
RUN = f'{RUNS}/unet_official_{PRECISION}'
resume = '--resume' if os.path.exists(f'{RUN}/checkpoints/last.pt') else ''
!python repro_magmap/scripts/train.py --out-dir {RUN} --device cuda --precision {PRECISION} {resume}

[data] 16 files in ['RFI-1'] under /content/BR/Data/RFI-Patch; 16 scenes; split train=12, val=2, test=2
[data] maps: train 2048, val 1024, test 8192 (test is not loaded during training); map (512, 512), normalization=map
[model] UNet 1->1, 9,502,561 parameters (authors' log: 9,502,561)
[train] precision requested=fp32 used=fp32; epochs=100; batch=4; lr=0.0002 AdamW wd=0.02; StepLR(50,0.5) initial_step=True; seed=1234
[check] sample tensors (1, 512, 512) torch.float32; input max 1.0000; label max 0.3432
[resume] from epoch 100 (lr now 5.000e-05)
[done] final checkpoint: checkpoints/epoch_100.pt; best val PSNR 38.319 (checkpoints/best_val.pt)


In [ ]:
# test evaluation: reported checkpoint (epoch 100) with supplementary image-domain metrics, then best-val
!python repro_magmap/scripts/evaluate.py --run {RUN} --device cuda --image-domain
!python repro_magmap/scripts/evaluate.py --run {RUN} --checkpoint best_val --device cuda

refusing to overwrite /content/drive/MyDrive/repro_magmap_runs/unet_official_fp32/eval_epoch_100
[data] 16 files in ['RFI-1'] under /content/BR/Data/RFI-Patch; 16 scenes; split train=12, val=2, test=2
[eval] best_val.pt (epoch 91), 8192 test maps, precision fp32, device cuda
  4/8192 maps, 2s
  804/8192 maps, 176s
  1604/8192 maps, 347s
  2404/8192 maps, 519s
  3204/8192 maps, 691s
  4004/8192 maps, 860s
  4804/8192 maps, 1032s
  5604/8192 maps, 1206s
  6404/8192 maps, 1378s
  7204/8192 maps, 1547s
  8004/8192 maps, 1718s

  SIR    input PSNR -> output PSNR | input SSIM -> output SSIM   (paper U-Net PSNR/SSIM read off Fig. 9)
      0  23.07+-4.50 -> 43.70+-3.48 | 0.9357 -> 0.9934   (30.7/0.965)
    -10  19.96+-2.53 -> 41.81+-3.86 | 0.8622 -> 0.9892   (30.8/0.948)
    -20  17.16+-2.08 -> 39.25+-4.06 | 0.6788 -> 0.9780   (31.5/0.953)
    -30  13.70+-2.14 -> 34.52+-4.27 | 0.4035 -> 0.9346   (25.4/0.852)
[eval] wrote /content/drive/MyDrive/repro_magmap_runs/unet_official_fp32/eval_best_val